## 0. Data preprocess

##### Since dataset contains many different types of features, require preprocess the data.

In [1]:
# import useful library
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt
from category_encoders import BinaryEncoder

from scipy.stats import kstest, norm

In [2]:
# import datasets
train_data = pd.read_csv("../COMP30027_2024_asst2_data/train_dataset.csv")
test_data = pd.read_csv("../COMP30027_2024_asst2_data/test_dataset.csv")


# count CountVectorizer features for director names, actor names, and possibly other text features
cv_path = "../COMP30027_2024_asst2_data/features_countvec/"
cvtrain_director_name = np.load(cv_path + "train_countvec_features_director_name.npy")
cvtrain_actor_1_name = np.load(cv_path + "train_countvec_features_actor_1_name.npy")
cvtrain_actor_2_name = np.load(cv_path + "train_countvec_features_actor_2_name.npy")

cvtest_director_name = np.load(cv_path + "test_countvec_features_director_name.npy")
cvtest_actor_1_name = np.load(cv_path + "test_countvec_features_actor_1_name.npy")
cvtest_actor_2_name = np.load(cv_path + "test_countvec_features_actor_2_name.npy")

# Doc2Vec features for plot keywords, genres, and other text features
d2v_path = "../COMP30027_2024_asst2_data/features_doc2vec/"
d2vtrain_plot = np.load(d2v_path + "train_doc2vec_features_plot_keywords.npy")
d2vtrain_genre = np.load(d2v_path + "train_doc2vec_features_genre.npy")
d2vtest_plot = np.load(d2v_path + "test_doc2vec_features_plot_keywords.npy")
d2vtest_genre = np.load(d2v_path + "test_doc2vec_features_genre.npy")

# FastText embeddings for movie titles
ft_path = "../COMP30027_2024_asst2_data/features_fasttext/"
fttrain_title = np.load(ft_path + "train_fasttext_title_embeddings.npy")
fttest_title = np.load(ft_path + "test_fasttext_title_embeddings.npy")

In [3]:
# convert the numpy array to dataframes
def convert_to_dataframe(np_array, prefix, index):
    # convert nparray to df
    col_names = [f"{prefix}_feat_{i}" for i in range(np_array.shape[1])]
    return pd.DataFrame(np_array, columns=col_names, index=index)

df_cvtrain_director_name = convert_to_dataframe(cvtrain_director_name, 'director', train_data.index)
df_cvtrain_actor_1_name = convert_to_dataframe(cvtrain_actor_1_name, 'actor_1', train_data.index)
df_cvtrain_actor_2_name = convert_to_dataframe(cvtrain_actor_2_name, 'actor_2', train_data.index)

df_d2vtrain_plot = convert_to_dataframe(d2vtrain_plot, 'plot_keywords', train_data.index)
df_d2vtrain_genre = convert_to_dataframe(d2vtrain_genre, 'genres', train_data.index)

df_fttrain_title = convert_to_dataframe(fttrain_title, 'title', train_data.index)

df_cvtest_director_name = convert_to_dataframe(cvtest_director_name, 'director', test_data.index)
df_cvtest_actor_1_name = convert_to_dataframe(cvtest_actor_1_name, 'actor_1', test_data.index)
df_cvtest_actor_2_name = convert_to_dataframe(cvtest_actor_2_name, 'actor_2', test_data.index)

df_d2vtest_plot = convert_to_dataframe(d2vtest_plot, 'plot_keywords', test_data.index)
df_d2vtest_genre = convert_to_dataframe(d2vtest_genre, 'genres', test_data.index)

df_fttest_title = convert_to_dataframe(fttest_title, 'title', test_data.index)

As the dataset contains missing values, fill it with a reasonable guess.

In [4]:
# handle missing values
missing_counts = train_data.isnull().sum()
print(missing_counts)
missing_row = train_data[train_data['language'].isnull()]
# as the language of a movie called "Labor Day" that the country is USA, good chance English
train_data.loc[2787, 'language'] = 'English'

id                           0
director_name                0
num_critic_for_reviews       0
duration                     0
director_facebook_likes      0
actor_3_facebook_likes       0
actor_2_name                 0
actor_1_facebook_likes       0
gross                        0
genres                       0
actor_1_name                 0
movie_title                  0
num_voted_users              0
cast_total_facebook_likes    0
actor_3_name                 0
facenumber_in_poster         0
plot_keywords                0
num_user_for_reviews         0
language                     0
country                      0
content_rating               0
title_year                   0
actor_2_facebook_likes       0
movie_facebook_likes         0
title_embedding              0
average_degree_centrality    0
imdb_score_binned            0
dtype: int64


From further inspect, the dataset also contains quite a bit 0 in the facebook like.\
Those are likely be missing values, and can alter the result quite a bit.\
Test the distribution of these column to decide what value should fill.

In [5]:
# director_facebook_likes
print((train_data['director_facebook_likes'] == 0).sum())

modified_data = train_data['director_facebook_likes'].replace(0, np.nan)
filtered_data = modified_data.dropna()

stat, p = kstest(filtered_data, 'norm', args=(filtered_data.mean(), filtered_data.std()))
print('k-stat:', stat, 'p-value:', p)

# non normal distribution, use median to fill all 0.
median = train_data['director_facebook_likes'][train_data['director_facebook_likes'] != 0].median()
train_data['director_facebook_likes'] = train_data['director_facebook_likes'].replace(0, median)

median = test_data['director_facebook_likes'][test_data['director_facebook_likes'] != 0].median()
test_data['director_facebook_likes'] = test_data['director_facebook_likes'].replace(0, median)

523
k-stat: 0.43482006434589937 p-value: 0.0


In [6]:
# movie_facebook_likes
print((train_data['movie_facebook_likes'] == 0).sum())

modified_data = train_data['movie_facebook_likes'].replace(0, np.nan)
filtered_data = modified_data.dropna()

stat, p = kstest(filtered_data, 'norm', args=(filtered_data.mean(), filtered_data.std()))
print('k-stat:', stat, 'p-value:', p)

# non normal distribution, use median to fill all 0.
median_value = train_data['movie_facebook_likes'][train_data['movie_facebook_likes'] != 0].median()
train_data['movie_facebook_likes'] = train_data['movie_facebook_likes'].replace(0, median_value)

median_value = test_data['movie_facebook_likes'][test_data['movie_facebook_likes'] != 0].median()
test_data['movie_facebook_likes'] = test_data['movie_facebook_likes'].replace(0, median_value)

1379
k-stat: 0.2497691636984521 p-value: 8.382326984770082e-90


The other numerical dataset contains minimum to no 0's, likely not missing.

#### 0.1 a dataset with all features with given embedding.
A simple dataset that just put all data together and stuff into models. \
This dataset likely not optimal, since only do it for training data and experiment with it.

In [7]:
train_full = train_data

In [8]:
train_full = pd.concat([train_full.drop(columns=['director_name','actor_1_name', 'actor_2_name']), df_cvtrain_director_name, df_cvtrain_actor_1_name, df_cvtrain_actor_2_name], axis=1)
train_full = pd.concat([train_full.drop(columns=['plot_keywords','genres']), df_d2vtrain_plot, df_d2vtrain_genre], axis=1)
train_full = pd.concat([train_full.drop(columns=['title_embedding']), df_fttrain_title], axis=1)

In [9]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# encoding
columns_to_encode = ['language', 'country', 'content_rating']

encoded_data = encoder.fit_transform(train_full[columns_to_encode])

encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(columns_to_encode))

# replace
train_full = train_full.drop(columns=columns_to_encode)

train_full = pd.concat([train_full, encoded_df], axis=1)

In [10]:
train_full = train_full.drop(columns=['movie_title', 'actor_3_name'])

In [11]:
train_full.to_csv("../COMP30027_2024_asst2_data/train_high_dim.csv", index=False)

#### 0.2 Numerical dataet for distance calculation based classification.
With above massive dimension, it is impossible to do algorithm that require distance calculation,
like knn, svm. We need a dataset with small dimension and meaningful data. \
Simply remove all text feature to experiment. This is naive approach.

In [12]:
train_notext = train_data
test_notext = test_data

In [13]:
# as those columns contain quite a lot unique values, and they are text based which is hard to process
# generate a train set that just discard them
# by logic, movie genre and plot_keyword should not interfere with rating, test the accuracy if drop them
columns_to_drop = ['director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name', 'movie_title', 'genres', 'plot_keywords', 'title_embedding', 'language', 'country', 'content_rating']
train_notext = train_notext.drop(columns=columns_to_drop)
test_notext = test_notext.drop(columns=columns_to_drop)

train_notext.to_csv("../COMP30027_2024_asst2_data/train_dataset_no_text.csv", index=False)
test_notext.to_csv("../COMP30027_2024_asst2_data/test_dataset_no_text.csv", index=False)

As all data is numerical, we can use f test to see the strength of each feature. \
From result, we can see the p-value is < 0.05, suggesting a statistically strong relation with change in response variable,
the label.

In [14]:
# split xy
numerical_cols = train_notext.columns.drop(['id', 'imdb_score_binned'])
X = train_notext[numerical_cols]
y = train_notext['imdb_score_binned']

# ANOVA F-test
f_values, p_values = f_classif(X, y)

results = pd.DataFrame({
    'Feature': numerical_cols,
    'F-value': f_values,
    'p-value': p_values
}).sort_values(by='p-value')

print(results)

                      Feature     F-value        p-value
6             num_voted_users  530.075648   0.000000e+00
9        num_user_for_reviews  151.709849  2.505394e-118
1                    duration  103.584817   9.685395e-83
0      num_critic_for_reviews   89.122897   1.147663e-71
12       movie_facebook_likes   89.045023   1.318056e-71
5                       gross   42.651752   7.601010e-35
2     director_facebook_likes   33.876191   1.094646e-27
10                 title_year   27.466067   2.005347e-22
13  average_degree_centrality   24.897300   2.616387e-20
7   cast_total_facebook_likes    7.431214   5.935612e-06
11     actor_2_facebook_likes    6.170993   6.078697e-05
4      actor_1_facebook_likes    5.728439   1.366543e-04
8        facenumber_in_poster    5.323933   2.854103e-04
3      actor_3_facebook_likes    4.541069   1.171562e-03


#### 0.3 full dataset with feature engineering and feature selection.
Since the full dataset contains a lot of dimensions, the massive features are actually causing the accuracy to go down,
hence feature engineering is necessary.

In [15]:
train_f = train_data
test_f = test_data

##### Let's deal with columns one by one.
First we have director name, as this is unique for each movie, categorize the director that produced 5 or less movies into one can help with noise reduction, as director with few entry can hardly draw meaningful conclusion, and reduce the risk of overfitting the model.

In [16]:
director_counts = train_f['director_name'].value_counts()
director_high = director_counts[director_counts > 5]

train_f['director_name'] = train_f['director_name'].apply(lambda x: x if x in director_high else "0_Other")

Do the same to the other names column.

In [17]:
actor_1_counts = train_f['actor_1_name'].value_counts()
actor_1_high = actor_1_counts[actor_1_counts > 5]
train_f['actor_1_name'] = train_f['actor_1_name'].apply(lambda x: x if x in actor_1_high else "0_Other")

actor_2_counts = train_f['actor_2_name'].value_counts()
actor_2_high = actor_2_counts[actor_2_counts > 5]
train_f['actor_2_name'] = train_f['actor_2_name'].apply(lambda x: x if x in actor_2_high else "0_Other")

actor_3_counts = train_f['actor_3_name'].value_counts()
actor_3_high = actor_3_counts[actor_3_counts > 5]
train_f['actor_3_name'] = train_f['actor_3_name'].apply(lambda x: x if x in actor_3_high else "0_Other")

In [18]:
# now encoding, use label encoding to preserve low dimension
le = LabelEncoder()
columns_to_encode = ['language', 'country', 'content_rating', 'director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name']

for column in columns_to_encode:
    train_f[column] = le.fit_transform(train_f[column])
    test_f[column] = le.fit_transform(test_f[column])

Now for the genres and plot_keyword, using multi-label-binarizer from scikit.\
Since many different plot_keyword, only use those with > 5 frequency.

In [19]:
# mlb = MultiLabelBinarizer()
# train_f['genres'] = train_f['genres'].apply(lambda x: x.split('|'))
# genres_encoded = mlb.fit_transform(train_f['genres'])
# genres_encoded_df = pd.DataFrame(genres_encoded, columns=mlb.classes_)

# train_f['plot_keywords'] = train_f['plot_keywords'].apply(lambda x: x.split('|'))
# plot_encoded = mlb.fit_transform(train_f['plot_keywords'])
# plot_encoded_df = pd.DataFrame(plot_encoded, columns=mlb.classes_)

train_f = train_f.drop(columns=['genres', 'plot_keywords'])
# train_f = pd.concat([train_f, genres_encoded_df, plot_encoded_df], axis=1)

# # test
# test_f['genres'] = test_f['genres'].apply(lambda x: x.split('|'))
# genres_encoded = mlb.fit_transform(test_f['genres'])
# genres_encoded_df = pd.DataFrame(genres_encoded, columns=mlb.classes_)

# test_f['plot_keywords'] = test_f['plot_keywords'].apply(lambda x: x.split('|'))
# plot_encoded = mlb.fit_transform(test_f['plot_keywords'])
# plot_encoded_df = pd.DataFrame(plot_encoded, columns=mlb.classes_)

test_f = test_f.drop(columns=['genres', 'plot_keywords'])
# test_f = pd.concat([test_f, genres_encoded_df, plot_encoded_df], axis=1)


For movie title, use the given array of embedding.

In [20]:
# train_f = pd.concat([train_f.drop(columns=['title_embedding', 'movie_title']), df_fttrain_title], axis=1)
train_f = train_f.drop(columns=['title_embedding', 'movie_title'])
test_f = test_f.drop(columns=['title_embedding', 'movie_title'])

In [22]:
print(train_f['imdb_score_binned'].value_counts())
for i in 

2    1839
3     777
1     235
4     129
0      24
Name: imdb_score_binned, dtype: int64


In [21]:
# output as csv
train_f.to_csv("../COMP30027_2024_asst2_data/train_select.csv", index=False)
test_f.to_csv("../COMP30027_2024_asst2_data/test_select.csv", index=False)